# 仓库机器人 LoRA 训练（Colab T4）

运行时选择 **T4 GPU**。训练产物是一个几十 MB 的 LoRA 适配器。


In [ ]:
# 每次都强制拉最新代码，避免运行时里残留旧版 train_lora.py
%cd /content
!rm -rf gh-app-test
!git clone -q https://github.com/xu926862-cyber/gh-app-test.git
%cd gh-app-test/bot/train
!pip install -q -r requirements.txt

In [ ]:
import torch
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')

In [ ]:
!BASE_MODEL=Qwen/Qwen2.5-Coder-3B-Instruct EPOCHS=3 python train_lora.py
!ls -la out/lora-repo-bot 2>/dev/null || echo "!! 训练没有产出 out/lora-repo-bot，上面的报错才是真正原因"

In [ ]:
# 打包并下载适配器：不依赖当前工作目录，运行时重启后也能看出问题在哪
import glob
import os
import zipfile

from google.colab import files

base = "/content/gh-app-test/bot/train"
print("cwd:", os.getcwd())
adapters = glob.glob("/content/**/adapter_model.safetensors", recursive=True)
print("找到的适配器:", adapters)

if not adapters:
    print("没有训练产物：要么训练单元没成功，要么运行时重启后 /content 被清空了。先重跑前面的单元。")
else:
    zip_path = os.path.join(base, "lora-repo-bot.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in adapters:
            root = os.path.dirname(f)
            for name in os.listdir(root):
                p = os.path.join(root, name)
                if os.path.isfile(p):
                    z.write(p, os.path.join(os.path.basename(root), name))
    print("打包完成:", zip_path, os.path.getsize(zip_path), "bytes")
    files.download(zip_path)